# xgb + lgb

In [1]:
import pandas as pd

In [2]:
from dataengineers import Dataset

In [3]:
dataset = Dataset('train')
train = dataset.build_main()

In [4]:
exclude = ['id', 'target', 'delivery_start']

In [5]:
features = [c for c in train.columns if c not in exclude]

In [6]:
ds2 = Dataset('test')
df_out = ds2.build_main()

## lgbm

In [7]:
from models import lGBM

In [8]:
lg = lGBM(features)

In [9]:
lg.fit(train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003031 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 18390
[LightGBM] [Info] Number of data points in the train set: 22103, number of used features: 83
[LightGBM] [Info] Start training from score 52.114479
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

In [10]:
lg_vals = lg.predict(df_out)

## XGB

In [11]:
from models import XGB

In [12]:
xg = XGB(features)

In [13]:
xg.fit(train)

In [14]:
xg_vals = xg.predict(df_out)

/home/matt/repos/nitor-comp/.venv/lib/python3.14/site-packages/xgboost/core.py:751: UserWarning: [21:46:30] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


## submission

In [15]:
from utils import Ensemble

In [16]:
ensem = Ensemble([0.5, 0.5], xg_vals, lg_vals)

In [17]:
y_out = ensem.build()

In [18]:
len(y_out)

13098

In [19]:
df_out['target'] = y_out

In [20]:
df_out.head()

,id,global_horizontal_irradiance,diffuse_horizontal_irradiance,direct_normal_irradiance,cloud_cover_total,cloud_cover_low,cloud_cover_mid,cloud_cover_high,precipitation_amount,visibility,...,load_forecast_lag_12h,load_forecast_lag_24h,load_forecast_lag_48h,load_forecast_lag_168h,load_forecast_rolling_mean_6h,load_forecast_rolling_mean_24h,load_forecast_ramp_1h,load_forecast_ramp_3h,load_forecast_ramp_6h,target
0,133627,0.0,0.0,0.0,100.0,14.0,44.0,100.0,0.0,16600.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.744660
1,133635,0.0,0.0,0.0,100.0,100.0,100.0,88.0,0.0,13800.0,...,NaN,NaN,NaN,NaN,55189.474200,55189.474200,-2203.0271,NaN,NaN,59.612690
2,133643,0.0,0.0,0.0,100.0,70.0,100.0,100.0,0.0,19700.0,...,NaN,NaN,NaN,NaN,54087.960650,54087.960650,-1707.9816,NaN,NaN,130.837325
3,133651,0.0,0.0,0.0,99.0,13.0,99.0,94.0,0.0,16200.0,...,NaN,NaN,NaN,NaN,53151.462267,53151.462267,-893.3085,-4804.3172,NaN,144.749542
4,133659,0.0,0.0,0.0,96.0,0.0,96.0,79.0,0.0,14500.0,...,NaN,NaN,NaN,NaN,52459.885950,52459.885950,-439.4928,-3040.7829,NaN,149.613300


In [21]:
from utils import Submission

In [22]:
submit = Submission(df_out)

In [23]:
submit.process()

,id,target
0,133627,109.744660
2183,133629,54.588298
4366,133630,50.277920
10915,133631,47.280004
6549,133633,46.347913
...,...,...
4365,146774,68.762702
6548,146775,37.087685
13097,146776,32.579254
8731,146777,38.230801


In [24]:
submit.validate()

✅ Validation passed!


In [25]:
submit.dump()